# Amazon Electronics Recommender
**Dataset:** Amazon Electronics Reviews (Kaggle)  
**Technique:** Matrix Factorization via Truncated SVD + Cosine Similarity  
**Author:** Portfolio Project — Amazon ML Summer School Application

---
### Pipeline Overview
1. Load & inspect the raw ratings data
2. Filter to active users only (reduces sparsity and saves RAM)
3. Build a User-Item sparse matrix
4. Compress it with Truncated SVD
5. Export the trained artefacts with Pickle

## Cell 1 — Imports

In [9]:
import pandas as pd
import numpy as np
import pickle

from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

print('All libraries loaded successfully.')

All libraries loaded successfully.


## Cell 2 — Load the Dataset
The raw CSV has **no header row**, so we manually assign column names.
We immediately drop `timestamp` because it carries no signal for collaborative filtering.

In [10]:
# The CSV has no headers, so we name the columns ourselves.
column_names = ['user_id', 'product_id', 'rating', 'timestamp']

df = pd.read_csv(
    'ratings_Electronics (1).csv',
    names=column_names,   # assign our own column names
    header=None           # tell pandas there is no header row in the file
)

# Drop timestamp — we only need who rated what, and how much.
df.drop(columns=['timestamp'], inplace=True)

print(f'Raw dataset shape: {df.shape}')
print(f'\nSample rows:')
df.head()

Raw dataset shape: (5327309, 3)

Sample rows:


,user_id,product_id,rating
0,NaN,B0029U2YSA,5.0
1,A19AL8BAS9CMZ,B0029U2YSA,4.0
2,A1UEITZI96EDK4,B0029U2YSA,5.0
3,A137IIOB056X8I,B0029U2YSA,5.0
4,A6RQ347XDYMBK,B0029U2YSA,4.0


## Cell 3 — Exploratory Snapshot
A quick look at rating distributions and dataset scale before we filter.

In [11]:
total_users    = df['user_id'].nunique()
total_products = df['product_id'].nunique()
total_ratings  = len(df)

print(f'Total unique users   : {total_users:,}')
print(f'Total unique products: {total_products:,}')
print(f'Total ratings        : {total_ratings:,}')
print(f'\nRating distribution:')
print(df['rating'].value_counts().sort_index())

Total unique users   : 3,169,199
Total unique products: 338,273
Total ratings        : 5,327,309

Rating distribution:
rating
1.0     598870
2.0     309034
3.0     434189
4.0     998318
5.0    2986898
Name: count, dtype: int64


## Cell 4 — Filter to Active Users
**Why?**  
The full dataset has millions of users who rated only 1–2 products.  
These "cold-start" users add almost no collaborative signal but massively inflate the matrix size.  
Keeping only users with **≥ 50 ratings** dramatically cuts memory usage while retaining the most
informative rows.

In [12]:
MIN_RATINGS_PER_USER = 50

# Count how many products each user has rated.
ratings_per_user = df.groupby('user_id')['product_id'].count()

# Keep only users who meet the minimum threshold.
active_users = ratings_per_user[ratings_per_user >= MIN_RATINGS_PER_USER].index

# Filter the main dataframe to those active users.
df_filtered = df[df['user_id'].isin(active_users)].copy()

print(f'Users  before filtering: {df["user_id"].nunique():,}')
print(f'Users  after  filtering: {df_filtered["user_id"].nunique():,}')
print(f'Rows   before filtering: {len(df):,}')
print(f'Rows   after  filtering: {len(df_filtered):,}')

Users  before filtering: 3,169,199
Users  after  filtering: 627
Rows   before filtering: 5,327,309
Rows   after  filtering: 49,335


## Cell 5 — Build the User-Item Matrix
We pivot the dataframe so that:
- Each **row** = one user
- Each **column** = one product
- Each **cell value** = the star rating (0 if not rated)

We then convert it to a **sparse matrix** (CSR format) so Python doesn't store millions of zeros in RAM.

In [14]:
# Pivot: rows = users, columns = products, values = ratings.
# fill_value=0 means "user hasn't rated this product".
user_item_matrix = df_filtered.pivot_table(
    index='user_id',
    columns='product_id',
    values='rating',
    fill_value=0
)

print(f'User-Item matrix shape: {user_item_matrix.shape}')
print(f'(rows = users, columns = products)')

# Save the product list BEFORE converting to sparse — we need the column labels later.
product_list = list(user_item_matrix.columns)
print(f'\nTotal products in matrix: {len(product_list):,}')

# Convert to a scipy sparse matrix to save memory.
# CSR (Compressed Sparse Row) is efficient for row-wise operations.
sparse_matrix = csr_matrix(user_item_matrix.values)

print(f'\nSparse matrix stored (zeros not held in RAM).')

User-Item matrix shape: (627, 18286)
(rows = users, columns = products)

Total products in matrix: 18,286

Sparse matrix stored (zeros not held in RAM).


## Cell 6 — Train the SVD Model
**What is Truncated SVD?**  
SVD decomposes our huge User-Item matrix into three smaller matrices.  
By keeping only the top `n_components=10` "latent factors", we:
1. Reduce noise (random one-off ratings get discarded)
2. Discover hidden patterns (e.g., users who like gaming keyboards also like gaming mice)
3. Compress the matrix so cosine similarity is fast to compute

In [15]:
N_COMPONENTS = 10  # number of latent factors to keep

svd_model = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)

# fit_transform learns the SVD decomposition AND returns the compressed matrix.
# The result shape is: (num_products, n_components)
# We transpose the sparse matrix so that each ROW = one product (not one user).
# This way, we'll later compare products to products using cosine similarity.
svd_matrix = svd_model.fit_transform(sparse_matrix.T)

print(f'Original sparse matrix (products x users): {sparse_matrix.T.shape}')
print(f'Compressed SVD matrix  (products x factors): {svd_matrix.shape}')

# How much of the original variance did we capture?
explained_variance = svd_model.explained_variance_ratio_.sum()
print(f'\nVariance explained by {N_COMPONENTS} components: {explained_variance:.2%}')

Original sparse matrix (products x users): (18286, 627)
Compressed SVD matrix  (products x factors): (18286, 10)

Variance explained by 10 components: 8.22%


## Cell 7 — Export Artefacts with Pickle
We save two objects:
- `svd_matrix.pkl` — the compressed product embeddings (shape: products × 10 factors)
- `product_list.pkl` — the ordered list of product IDs (so we can map matrix rows back to real IDs)

In [16]:
# Save the SVD matrix (the compressed product embeddings)
with open('svd_matrix.pkl', 'wb') as f:
    pickle.dump(svd_matrix, f)

# Save the product list (needed to look up which row = which product)
with open('product_list.pkl', 'wb') as f:
    pickle.dump(product_list, f)

print('Artefacts exported:')
print('  svd_matrix.pkl   — shape:', svd_matrix.shape)
print('  product_list.pkl — total products:', len(product_list))

Artefacts exported:
  svd_matrix.pkl   — shape: (18286, 10)
  product_list.pkl — total products: 18286


## Cell 8 — Quick Sanity Check
Before building the app, verify the pipeline end-to-end by loading the pickles
and running a sample recommendation query.

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

# --- Load saved artefacts ---
with open('svd_matrix.pkl', 'rb') as f:
    loaded_svd_matrix = pickle.load(f)

with open('product_list.pkl', 'rb') as f:
    loaded_product_list = pickle.load(f)

# --- Pick a test product (the first one in our list) ---
test_product_id = loaded_product_list[0]
test_product_index = 0

# --- Compute cosine similarity between this product and all others ---
# cosine_similarity expects a 2D array, so we use [test_product_index] to get one row.
similarity_scores = cosine_similarity(
    [loaded_svd_matrix[test_product_index]],
    loaded_svd_matrix
).flatten()

# --- Rank products by similarity score (descending) ---
# argsort gives ascending order, so we reverse it with [::-1]
ranked_indices = similarity_scores.argsort()[::-1]

# Skip index 0 (the product itself) and take the next 5
top_5_indices = [i for i in ranked_indices if i != test_product_index][:5]
top_5_products = [loaded_product_list[i] for i in top_5_indices]

print(f'Input product  : {test_product_id}')
print(f'\nTop 5 recommended products:')
for rank, pid in enumerate(top_5_products, start=1):
    score = similarity_scores[loaded_product_list.index(pid)]
    print(f'  {rank}. {pid}  (similarity: {score:.4f})')

Input product  : B0029U2YSA

Top 5 recommended products:
  1. B004US0URQ  (similarity: 1.0000)
  2. B004ZL2OB8  (similarity: 1.0000)
  3. B003BI6W0K  (similarity: 1.0000)
  4. B004MJ9MZE  (similarity: 1.0000)
  5. B0078GCLUG  (similarity: 1.0000)
